# 11.1.4 Chunking

긴 문서를 `RecursiveCharacterTextSplitter`로 일정한 크기의 조각(chunk)으로 나누는 예제입니다.

Chunking은 긴 문서를 embedding, 검색, RAG, 요약 등에 사용하기 전에 문맥을 유지하면서 적당한 길이로 나누는 과정입니다.

이 노트북의 흐름:

1. Text splitter import
2. 샘플 긴 문서 준비
3. 기본 chunking 실행
4. overlap이 있는 chunking 비교
5. chunk 결과를 embedding 입력 형태로 변환


## 1. 패키지 준비

현재 환경에서는 최신 패키지인 `langchain_text_splitters`를 사용합니다.

예전 자료에서는 `from langchain.text_splitter import RecursiveCharacterTextSplitter` 형태를 쓰지만, 최신 LangChain에서는 분리된 패키지 경로를 권장합니다.

In [1]:
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

print("RecursiveCharacterTextSplitter ready")

RecursiveCharacterTextSplitter ready


## 2. 샘플 문서 준비

실습용으로 부동산 경기 전망 기사처럼 여러 문단으로 된 긴 텍스트를 준비합니다.

In [2]:
text = """
수도권과 비수도권의 주택사업 경기 전망이 엇갈리고 있다. 주택사업자들은 수도권 주택 경기가 전월 대비 나아질 것으로 판단하는 반면, 비수도권은 침체 흐름이 이어질 가능성이 크다고 보고 있다.

주택산업연구원이 주택사업자를 대상으로 조사한 결과, 전국 주택사업 경기 전망 지수는 전월 대비 하락했다. 수도권은 소폭 상승했지만, 비수도권은 전반적으로 하락세를 보였다. 이 지수는 기준선인 100을 넘으면 경기가 좋아질 것으로 보는 업체 비율이 높다는 의미이고, 100을 밑돌면 반대 의미로 해석된다.

수도권 가운데 서울은 공급 부족 우려와 가격 상승 기대가 겹치면서 여전히 높은 수준을 유지했다. 인천과 경기 일부 지역도 회복 기대가 나타났지만, 지역별 편차는 컸다. 금리 인하 기대와 청약 시장 회복 흐름이 수도권 심리에 긍정적으로 작용한 것으로 분석된다.

비수도권의 경우 대출 규제, 금리 부담, 미분양 증가 등이 사업자 심리에 부정적인 영향을 주고 있다. 광역시와 지방 중소도시 모두 하락세가 나타났으며, 일부 지역에서는 주택 가격 회복이 더딘 상황이다.

전문가들은 지역별 수요와 공급 상황을 구분해 분석해야 한다고 조언한다. 같은 주택시장이라도 수도권 핵심 지역과 지방 외곽 지역의 시장 여건이 크게 다르기 때문이다.
""".strip()

print("문자 수:", len(text))
print(text[:200])

문자 수: 630
수도권과 비수도권의 주택사업 경기 전망이 엇갈리고 있다. 주택사업자들은 수도권 주택 경기가 전월 대비 나아질 것으로 판단하는 반면, 비수도권은 침체 흐름이 이어질 가능성이 크다고 보고 있다.

주택산업연구원이 주택사업자를 대상으로 조사한 결과, 전국 주택사업 경기 전망 지수는 전월 대비 하락했다. 수도권은 소폭 상승했지만, 비수도권은 전반적으로 하락세를 보


## 3. 기본 Chunking

`chunk_size=256`, `chunk_overlap=0`으로 설정하면 각 chunk가 최대 256자 안팎이 되도록 문서를 나눕니다.

`RecursiveCharacterTextSplitter`는 문단, 줄바꿈, 문장, 단어 순서로 최대한 자연스럽게 나누려고 시도합니다.

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=256,
    chunk_overlap=0,
)

docs = text_splitter.create_documents([text])

print("chunk 개수:", len(docs))

for idx, doc in enumerate(docs, start=1):
    print(f"Chunk {idx} / length={len(doc.page_content)}")
    print(doc.page_content)
    print("-" * 80)

chunk 개수: 4
Chunk 1 / length=106
수도권과 비수도권의 주택사업 경기 전망이 엇갈리고 있다. 주택사업자들은 수도권 주택 경기가 전월 대비 나아질 것으로 판단하는 반면, 비수도권은 침체 흐름이 이어질 가능성이 크다고 보고 있다.
--------------------------------------------------------------------------------
Chunk 2 / length=169
주택산업연구원이 주택사업자를 대상으로 조사한 결과, 전국 주택사업 경기 전망 지수는 전월 대비 하락했다. 수도권은 소폭 상승했지만, 비수도권은 전반적으로 하락세를 보였다. 이 지수는 기준선인 100을 넘으면 경기가 좋아질 것으로 보는 업체 비율이 높다는 의미이고, 100을 밑돌면 반대 의미로 해석된다.
--------------------------------------------------------------------------------
Chunk 3 / length=144
수도권 가운데 서울은 공급 부족 우려와 가격 상승 기대가 겹치면서 여전히 높은 수준을 유지했다. 인천과 경기 일부 지역도 회복 기대가 나타났지만, 지역별 편차는 컸다. 금리 인하 기대와 청약 시장 회복 흐름이 수도권 심리에 긍정적으로 작용한 것으로 분석된다.
--------------------------------------------------------------------------------
Chunk 4 / length=205
비수도권의 경우 대출 규제, 금리 부담, 미분양 증가 등이 사업자 심리에 부정적인 영향을 주고 있다. 광역시와 지방 중소도시 모두 하락세가 나타났으며, 일부 지역에서는 주택 가격 회복이 더딘 상황이다.

전문가들은 지역별 수요와 공급 상황을 구분해 분석해야 한다고 조언한다. 같은 주택시장이라도 수도권 핵심 지역과 지방 외곽 지역의 시장 여건이 크게 다르기 때문이다.
---------------------------------

## 4. Overlap이 있는 Chunking

`chunk_overlap`은 이전 chunk의 끝부분을 다음 chunk에 일부 반복해서 넣는 설정입니다.

검색이나 RAG에서는 경계 부분의 문맥이 잘리는 것을 줄이기 위해 overlap을 사용하는 경우가 많습니다.

In [4]:
overlap_splitter = RecursiveCharacterTextSplitter(
    chunk_size=256,
    chunk_overlap=40,
)

overlap_docs = overlap_splitter.create_documents([text])

print("overlap chunk 개수:", len(overlap_docs))

for idx, doc in enumerate(overlap_docs, start=1):
    print(f"Chunk {idx} / length={len(doc.page_content)}")
    print(doc.page_content)
    print("-" * 80)

overlap chunk 개수: 4
Chunk 1 / length=106
수도권과 비수도권의 주택사업 경기 전망이 엇갈리고 있다. 주택사업자들은 수도권 주택 경기가 전월 대비 나아질 것으로 판단하는 반면, 비수도권은 침체 흐름이 이어질 가능성이 크다고 보고 있다.
--------------------------------------------------------------------------------
Chunk 2 / length=169
주택산업연구원이 주택사업자를 대상으로 조사한 결과, 전국 주택사업 경기 전망 지수는 전월 대비 하락했다. 수도권은 소폭 상승했지만, 비수도권은 전반적으로 하락세를 보였다. 이 지수는 기준선인 100을 넘으면 경기가 좋아질 것으로 보는 업체 비율이 높다는 의미이고, 100을 밑돌면 반대 의미로 해석된다.
--------------------------------------------------------------------------------
Chunk 3 / length=144
수도권 가운데 서울은 공급 부족 우려와 가격 상승 기대가 겹치면서 여전히 높은 수준을 유지했다. 인천과 경기 일부 지역도 회복 기대가 나타났지만, 지역별 편차는 컸다. 금리 인하 기대와 청약 시장 회복 흐름이 수도권 심리에 긍정적으로 작용한 것으로 분석된다.
--------------------------------------------------------------------------------
Chunk 4 / length=205
비수도권의 경우 대출 규제, 금리 부담, 미분양 증가 등이 사업자 심리에 부정적인 영향을 주고 있다. 광역시와 지방 중소도시 모두 하락세가 나타났으며, 일부 지역에서는 주택 가격 회복이 더딘 상황이다.

전문가들은 지역별 수요와 공급 상황을 구분해 분석해야 한다고 조언한다. 같은 주택시장이라도 수도권 핵심 지역과 지방 외곽 지역의 시장 여건이 크게 다르기 때문이다.
-------------------------

## 5. 문자 기준 Fixed Chunking

문단 기준을 거의 쓰지 않고 고정 길이에 가깝게 자르고 싶다면 `separators=[""]`를 사용할 수 있습니다.

이 방식은 단순하지만 문장이나 단어 중간에서 잘릴 수 있습니다.

In [5]:
fixed_splitter = RecursiveCharacterTextSplitter(
    chunk_size=256,
    chunk_overlap=20,
    separators=[""],
)

fixed_docs = fixed_splitter.create_documents([text])

print("fixed chunk 개수:", len(fixed_docs))

for idx, doc in enumerate(fixed_docs, start=1):
    print(f"Chunk {idx} / length={len(doc.page_content)}")
    print(doc.page_content)
    print("-" * 80)

fixed chunk 개수: 3
Chunk 1 / length=255
수도권과 비수도권의 주택사업 경기 전망이 엇갈리고 있다. 주택사업자들은 수도권 주택 경기가 전월 대비 나아질 것으로 판단하는 반면, 비수도권은 침체 흐름이 이어질 가능성이 크다고 보고 있다.

주택산업연구원이 주택사업자를 대상으로 조사한 결과, 전국 주택사업 경기 전망 지수는 전월 대비 하락했다. 수도권은 소폭 상승했지만, 비수도권은 전반적으로 하락세를 보였다. 이 지수는 기준선인 100을 넘으면 경기가 좋아질 것으로 보는 업체 비율이 높다는 의미이고,
--------------------------------------------------------------------------------
Chunk 2 / length=256
보는 업체 비율이 높다는 의미이고, 100을 밑돌면 반대 의미로 해석된다.

수도권 가운데 서울은 공급 부족 우려와 가격 상승 기대가 겹치면서 여전히 높은 수준을 유지했다. 인천과 경기 일부 지역도 회복 기대가 나타났지만, 지역별 편차는 컸다. 금리 인하 기대와 청약 시장 회복 흐름이 수도권 심리에 긍정적으로 작용한 것으로 분석된다.

비수도권의 경우 대출 규제, 금리 부담, 미분양 증가 등이 사업자 심리에 부정적인 영향을 주고 있다. 광역시와 지방 중소
--------------------------------------------------------------------------------
Chunk 3 / length=158
향을 주고 있다. 광역시와 지방 중소도시 모두 하락세가 나타났으며, 일부 지역에서는 주택 가격 회복이 더딘 상황이다.

전문가들은 지역별 수요와 공급 상황을 구분해 분석해야 한다고 조언한다. 같은 주택시장이라도 수도권 핵심 지역과 지방 외곽 지역의 시장 여건이 크게 다르기 때문이다.
--------------------------------------------------------------------------------


## 6. Embedding 입력 형태로 변환

Chroma DB에 저장하거나 embedding 모델에 넣을 때는 `Document` 객체의 `page_content`만 리스트로 추출해서 사용할 수 있습니다.

In [6]:
chunk_texts = [doc.page_content for doc in docs]

print("embedding 입력 개수:", len(chunk_texts))
print("첫 번째 chunk:")
print(chunk_texts[0])

embedding 입력 개수: 4
첫 번째 chunk:
수도권과 비수도권의 주택사업 경기 전망이 엇갈리고 있다. 주택사업자들은 수도권 주택 경기가 전월 대비 나아질 것으로 판단하는 반면, 비수도권은 침체 흐름이 이어질 가능성이 크다고 보고 있다.


## 7. 정리

- `chunk_size`는 각 chunk의 최대 길이를 조절합니다.
- `chunk_overlap`은 chunk 경계에서 문맥이 끊기는 문제를 줄입니다.
- `separators`를 조정하면 문단 중심, 문장 중심, 문자 중심 분할을 선택할 수 있습니다.
- RAG에서는 chunking 품질이 검색 품질과 답변 품질에 직접 영향을 줍니다.